> **TrustBreast — Notebook 4 of 5.** Tables 14–15; Figs 13–14. Needs `O3_all_patients.csv` from notebook 3 (also provided in `results/`).

# File 4 — FIXED version (Objective 4: DiCE counterfactuals)

## Chalane ka tareeqa
1. Colab → **Runtime → Change runtime type → CPU** (default; GPU nahi)
2. **Runtime → Run all**, Drive ki permission par **Allow**
3. **STEP 3** par file upload ka button aayega → File 3 ke zip se nikali hui **`O3_all_patients.csv`** chunein.
   (Ya pehle hi left side **Files** panel mein yeh CSV drag karke daal dein — phir button nahi aayega.)
4. Waqt: takreeban **10–20 minute**
5. **STEP 9 — RESULTS SUMMARY** ka output bhejein, aur aakhri cell ka zip bhi.

## Kya theek kiya gaya
- **Seed-lock:** DiCE ab `random_seed=42` ke saath chalta hai → har run par SAME counterfactuals (pehle har run alag aate the — paper ki Limitation).
- Counterfactuals **do dafa** banaye jate hain aur check hota hai ke dono bilkul same hain (reproducibility proof).
- Purane MC-Dropout "PREP" cells (unseeded, alag numbers dete the) hata diye — high-uncertainty patients seedha File 3 ki CSV se aate hain.
- **Threshold fix:** DiCE pehle 0.5 par class tay karta tha, jabke ensemble 0.35 par decide karta hai. Ab DiCE ki boundary 0.35 ke saath aligned hai.
- Saare counterfactuals + plausibility ek CSV mein save hote hain (GitHub ke liye).

⚠️ Seed lagane se counterfactuals **naye** honge, is liye Table 14, Table 15, Fig 13/14 aur L1 numbers is run ke output se update honge.

In [ ]:
# Colab: repo clone karo (models/ folder isi mein hai). Local Jupyter par yeh cell kuch nahi karta.
import os
if os.path.exists('/content') and not os.path.isdir('models') and not os.path.isdir('../models'):
    !git clone -q https://github.com/Iqra672-ai/TrustBreast.git /content/TrustBreast
    %cd /content/TrustBreast
    !pip -q install -r requirements.txt


## STEP 1 — Determinism + install + LOAD

In [ ]:
# ============================================
# CELL 0 — DETERMINISM  (SAB SE PEHLE chalao, imports se bhi pehle)
# Yeh 3 cheezein add karta hai jo DNN ko har run SAME banati hain:
#   1. os.environ flags   -> GPU/cuDNN ko deterministic (imports se PEHLE set hona zaroori)
#   2. saare seeds        -> python / numpy / tensorflow
#   3. enable_op_determinism() -> GPU floating-point order LOCK (yehi asal missing cheez thi)
# reseed() helper baad me DNN se theek pehle RNG ko wapas fix karta hai.
# ============================================
import os
os.environ['PYTHONHASHSEED']         = '42'
os.environ['TF_DETERMINISTIC_OPS']   = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import random, numpy as np, tensorflow as tf
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print("enable_op_determinism() ON  ->  DNN ab reproducible")
except Exception as e:
    print("Note: enable_op_determinism unavailable (purani TF). Baqi fixes phir bhi lagenge.")

def reseed(s=SEED):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)

print("TF:", tf.__version__, "| Determinism setup done. Ab baqi cells chalao.")


In [ ]:
!pip -q install scikit-learn xgboost imbalanced-learn tensorflow scipy shap lime dice-ml anthropic matplotlib seaborn

In [ ]:
# ===== LOAD the locked 99.12% model (run this FIRST) =====
# Requires the saved model folder in Google Drive: MyDrive/TrustBreast_locked/
# (produced once by File 1 - Objective 1). No retraining here, so the number is always 99.12%.
import os, pickle, joblib, numpy as np, tensorflow as tf
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)
# Locked model: pehle GitHub repo ka models/ folder, warna Google Drive
SAVE_DIR = next((p for p in ['models/TrustBreast_locked', '../models/TrustBreast_locked']
                 if os.path.isdir(p)), None)
if SAVE_DIR is None:
    SAVE_DIR = '/content/drive/MyDrive/TrustBreast_locked'
    from google.colab import drive; drive.mount('/content/drive')
print('Loading locked model from:', SAVE_DIR)
rf_model  = joblib.load(SAVE_DIR + '/rf_model.pkl')
xgb_model = joblib.load(SAVE_DIR + '/xgb_model.pkl')
scaler    = joblib.load(SAVE_DIR + '/scaler.pkl')
dnn_best  = tf.keras.models.load_model(SAVE_DIR + '/dnn_best.keras')
dnn_model = tf.keras.models.load_model(SAVE_DIR + '/dnn_model.keras')
with open(SAVE_DIR + '/state.pkl','rb') as f: state = pickle.load(f)
globals().update({k:v for k,v in state.items() if v is not None})
if globals().get('prob_ensemble_val') is None and 'X_val_sc' in globals():
    _rf=rf_model.predict_proba(X_val_sc)[:,1]; _xg=xgb_model.predict_proba(X_val_sc)[:,1]
    _dn=dnn_best.predict(X_val_sc, verbose=0).ravel(); prob_ensemble_val=(_rf+_xg+_dn)/3
model_dnn=dnn_best; rf_aug=rf_model; xgb_aug=xgb_model; feature_names=list(X.columns)
print('LOADED locked model. Ensemble accuracy:', round(accuracy_score(y_test, ens_pred)*100,2), 'percent')

In [ ]:
# --- aliases so O2/O3/O4 cells find the trained models ---
model_dnn = dnn_best        # O3 (MC Dropout) expects this name
rf_aug    = rf_model        # O4 (DiCE) expects this name
xgb_aug   = xgb_model       # O4 (DiCE) expects this name
feature_names = list(X.columns)
print('Bridge ready. Ensemble threshold from O1:', best_ens_thr)

## STEP 2 — Data, DiCE data object, ensemble wrapper, constraints

In [ ]:
import dice_ml
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Data load (as in previous complete refresh cells)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"
col_names = ['id','diagnosis',
  'radius_mean','texture_mean','perimeter_mean','area_mean',
  'smoothness_mean','compactness_mean','concavity_mean',
  'concave_points_mean','symmetry_mean','fractal_dimension_mean',
  'radius_se','texture_se','perimeter_se','area_se',
  'smoothness_se','compactness_se','concavity_se',
  'concave_points_se','symmetry_se','fractal_dimension_se',
  'radius_worst','texture_worst','perimeter_worst','area_worst',
  'smoothness_worst','compactness_worst','concavity_worst',
  'concave_points_worst','symmetry_worst','fractal_dimension_worst']

df = pd.read_csv(url, header=None, names=col_names)

le = LabelEncoder()
df['diagnosis'] = le.fit_transform(df['diagnosis'])

X = df.drop(['id','diagnosis'], axis=1)
y = df['diagnosis']

# Define feature_names
feature_names = X.columns.tolist()

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ============================================
# STEP 4.2 — CREATE DiCE DATA OBJECT
# ============================================

# Apna training data ek DataFrame mein combine karo (features + target)
df_train_dice = pd.DataFrame(X_train, columns=feature_names)  # feature_names = WBCD ke 30 column names
df_train_dice['diagnosis'] = y_train  # target column add karo (0=Benign, 1=Malignant)

print(f"DiCE training data shape: {df_train_dice.shape}")
print(df_train_dice.head())

# Continuous features list (WBCD ke saare 30 features continuous hain)
continuous_features = feature_names  # sab numeric measurements hain (radius, texture, etc.)

# DiCE Data object banao
d = dice_ml.Data(
    dataframe=df_train_dice,
    continuous_features=continuous_features,
    outcome_name='diagnosis'
)

print("\n✅ DiCE Data object created successfully!")
print(f"   Total features: {len(continuous_features)}")
print(f"   Outcome column: 'diagnosis'")

In [ ]:
import numpy as np

class EnsembleWrapper:
    """
    Wrapper jo DiCE ke unscaled input ko
    scale kar ke ensemble model ko deta hai
    """
    def __init__(self, rf_model, xgb_model, dnn_model, scaler, threshold=0.5, weights=None):
        self.rf_model = rf_model
        self.xgb_model = xgb_model
        self.dnn_model = dnn_model
        self.scaler = scaler
        self.threshold = threshold   # O1 ka best_ens_thr yahan aata hai
        # Agar tumne specific weights diye the soft-voting mein, yahan daalo
        self.weights = weights if weights else [1/3, 1/3, 1/3]

    def predict_proba(self, X):
        # X DiCE se aata hai (unscaled, DataFrame ya array)
        if hasattr(X, 'values'):
            X = X.values

        # Step 1: Scale karo
        X_scaled = self.scaler.transform(X)

        # Step 2: Har model se prediction lo
        rf_proba = self.rf_model.predict_proba(X_scaled)
        xgb_proba = self.xgb_model.predict_proba(X_scaled)
        dnn_proba = self.dnn_model.predict(X_scaled)  # DNN ka output shape check karna padega

        # Agar DNN sirf 1 column deta hai (sigmoid), to 2-column banao
        if dnn_proba.ndim == 1 or dnn_proba.shape[1] == 1:
            dnn_proba = np.hstack([1 - dnn_proba.reshape(-1,1), dnn_proba.reshape(-1,1)])

        # Step 3: Soft-voting (weighted average)
        final_proba = (self.weights[0] * rf_proba +
                       self.weights[1] * xgb_proba +
                       self.weights[2] * dnn_proba)

        return final_proba

    def predict(self, X):
        proba = self.predict_proba(X)
        return (proba[:, 1] >= self.threshold).astype(int)  # O1 best_ens_thr (0.5 nahi)


# NOTE: locked scaler ko refit NAHI karna — woh state.pkl se load hua hai.
assert hasattr(scaler, 'data_min_'), 'scaler fitted nahi — File 1 ka LOAD cell chalao'

# Wrapper object banao — apne actual trained models daalo yahan
ensemble_wrapper = EnsembleWrapper(
    rf_model=rf_aug,        # tumhara trained Random Forest
    xgb_model=xgb_aug,      # tumhara trained XGBoost
    dnn_model=dnn_best,      # tumhara trained DNN
    scaler=scaler,            # already fitted MinMaxScaler (locked)
    threshold=best_ens_thr   # ← O1 ka VALIDATION-tuned threshold (0.5 nahi)
)

# Quick test — kya wrapper sahi kaam kar raha hai?
test_sample = X_train.iloc[:2]  # 2 unscaled samples
test_proba = ensemble_wrapper.predict_proba(test_sample)
print("Test prediction (proba):", test_proba)

# ============================================
# DiCE Model Object Create Karo
# ============================================

m = dice_ml.Model(
    model=ensemble_wrapper,
    backend="sklearn",
    model_type="classifier"
)

print("\n✅ DiCE Model object created successfully!")

In [ ]:
# ★ FIX: DiCE ki decision boundary ko ensemble ke validated threshold (0.35) ke saath align karo.
# DiCE "opposite class" aur validity 0.5 par tay karta hai; hamara ensemble 0.35 par decide karta hai.
# Monotone piecewise-linear rescaling: p < t -> [0, 0.5),  p >= t -> [0.5, 1].  Is se p = 0.35 DiCE ke liye 0.5 ban jata hai.
# (Sirf DiCE ke andar; reported probabilities asli ensemble_wrapper se hi aati hain.)
import numpy as np, dice_ml
class ThresholdAlignedWrapper(EnsembleWrapper):
    def predict_proba(self, X):
        p = super().predict_proba(X)[:, 1]; t = self.threshold
        q = np.where(p < t, 0.5 * p / t, 0.5 + 0.5 * (p - t) / (1 - t))
        return np.column_stack([1 - q, q])
dice_wrapper = ThresholdAlignedWrapper(rf_model=rf_aug, xgb_model=xgb_aug, dnn_model=dnn_best,
                                       scaler=scaler, threshold=best_ens_thr)
m = dice_ml.Model(model=dice_wrapper, backend="sklearn", model_type="classifier")
print(f"DiCE boundary aligned to ensemble threshold {best_ens_thr:.2f}")


In [ ]:
import dice_ml, importlib.metadata
exp_dice = dice_ml.Dice(d, m, method="random")
permitted_range = {f: [float(X_train[f].min()), float(X_train[f].max())] for f in feature_names}
features_to_vary = feature_names.copy()
print("dice-ml version:", importlib.metadata.version('dice-ml'))
print(f"Permitted ranges (training min–max) set for {len(permitted_range)} features; all {len(features_to_vary)} may vary")


## STEP 3 — High-uncertainty patients (File 3 ki `O3_all_patients.csv` se)

In [ ]:
import os, glob, pandas as pd
cands = sorted(glob.glob('O3_all_patients*.csv') + glob.glob('results/O3_all_patients*.csv') + glob.glob('../results/O3_all_patients*.csv'))
if not cands:
    print("O3_all_patients.csv upload karein (File 3 ke zip se):")
    from google.colab import files; up = files.upload(); cands = sorted(k for k in up if k.startswith('O3_all_patients'))
df_mc = pd.read_csv(cands[0])
df_high = df_mc[df_mc['biopsy_flag']].sort_values('mc_std', ascending=False)
high_uncertainty_indices = df_high['patient_idx'].tolist()
print(f"Loaded {cands[0]} -> {len(df_high)} high-uncertainty patients: {high_uncertainty_indices}")
assert sorted(high_uncertainty_indices) == [3, 16, 18, 29, 42, 56, 61, 62, 79, 87, 111, 112], "CSV File 3 wali nahi lagti!"


## STEP 4 — ★ Seed-locked counterfactuals, DO dafa (reproducibility check)

In [ ]:
import numpy as np, random, pandas as pd
DICE_SEED = 42
def build_cache(seed=DICE_SEED):
    cache = {}
    for idx in high_uncertainty_indices:
        np.random.seed(seed + int(idx)); random.seed(seed + int(idx))
        res = exp_dice.generate_counterfactuals(
            X_test.iloc[[idx]], total_CFs=3, desired_class="opposite",
            permitted_range=permitted_range, features_to_vary=features_to_vary,
            random_seed=seed + int(idx))
        cache[idx] = res.cf_examples_list[0].final_cfs_df.reset_index(drop=True)
    return cache

cf_cache  = build_cache()
cf_cache2 = build_cache()
same = all(cf_cache[i] is not None and cf_cache[i].equals(cf_cache2[i]) for i in high_uncertainty_indices)
n_ok = sum(1 for v in cf_cache.values() if v is not None and len(v) > 0)
print(f"\nCFs found for {n_ok}/{len(high_uncertainty_indices)} patients | total CFs = {sum(len(v) for v in cf_cache.values())}")
print("DiCE REPRODUCIBLE ✅ (dono runs bilkul same)" if same else "DiCE NOT reproducible ❌")
DICE_REPRO = same


## STEP 5 — Per-patient changes (STEP 4.6)

In [ ]:
# ============================================
# STEP 4.6 — 3 COUNTERFACTUALS FOR 12 HIGH-UNCERTAINTY PATIENTS
# ============================================

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 12 high uncertainty patient indices df_high se nikalo
high_uncertainty_indices = df_high['patient_idx'].tolist()
print(f"Processing {len(high_uncertainty_indices)} patients: {high_uncertainty_indices}\n")

all_cf_results = []
feature_change_counter = {feat: 0 for feat in feature_names}
failed_patients = []

for idx in high_uncertainty_indices:
    print(f"\n{'='*60}")
    print(f"Patient {idx} — generating 3 counterfactuals...")
    print('='*60)

    try:
        # Query instance nikalo (positional index, jaise X_test.iloc)
        query_instance = X_test.iloc[[idx]]
        true_label = y_test.iloc[idx]
        current_pred = ensemble_wrapper.predict_proba(query_instance)[0][1]

        # CF cache se lo (dobara generate NAHI)
        cf_df = cf_cache.get(idx)

        if cf_df is None or len(cf_df) == 0:
            print(f"  ⚠️ No counterfactual found for patient {idx}")
            failed_patients.append(idx)
            continue

        # Original values se compare karo — kaunsi features change hui
        original_vals = query_instance.iloc[0]
        changed_features_this_patient = []

        for i, row in cf_df.iterrows():
            for feat in feature_names:
                orig_val = original_vals[feat]
                cf_val = row[feat]
                # Significant change check karo (>1% relative difference)
                if abs(cf_val - orig_val) > 0.001 * (permitted_range[feat][1] - permitted_range[feat][0]):
                    feature_change_counter[feat] += 1
                    changed_features_this_patient.append(feat)

        all_cf_results.append({
            'patient_idx': idx,
            'true_label': true_label,
            'mc_mean': df_high[df_high['patient_idx']==idx]['mc_mean'].values[0],
            'current_pred': current_pred,
            'n_cfs_found': len(cf_df),
            'changed_features': list(set(changed_features_this_patient))
        })

        print(f"  ✅ {len(cf_df)} CFs found. Features changed: {set(changed_features_this_patient)}")

    except Exception as e:
        print(f"  ❌ Error for patient {idx}: {str(e)}")
        failed_patients.append(idx)

# ============================================
# SUMMARY
# ============================================
print(f"\n{'='*60}")
print("STEP 4.6 SUMMARY")
print('='*60)
print(f"Successfully processed: {len(all_cf_results)}/{len(high_uncertainty_indices)} patients")
print(f"Failed: {failed_patients}")

# Feature importance summary — kaunsi feature sabse zyada change hui
feature_summary = pd.Series(feature_change_counter).sort_values(ascending=False)
print("\n🎯 Top 10 Most Frequently Changed Features (across all patients/CFs):")
print(feature_summary.head(10))

# Results dataframe banao
df_cf_summary = pd.DataFrame(all_cf_results)
print("\n📋 Per-Patient Summary:")
print(df_cf_summary[['patient_idx', 'true_label', 'mc_mean', 'changed_features']])

## STEP 6 — Plausibility filter (STEP 4.7)

In [ ]:
# ============================================
# STEP 4.7 — PLAUSIBILITY FILTERING
# ============================================

from sklearn.neighbors import NearestNeighbors
import numpy as np
import pandas as pd

# ----------------------------------------------
# 1️⃣ k-NN DISTANCE CHECK — Kya CF "real data" jaisa lagta hai?
# ----------------------------------------------
# Training data (scaled) pe NearestNeighbors fit karo
nn_model = NearestNeighbors(n_neighbors=5)
nn_model.fit(scaler.transform(X_train))

def check_knn_plausibility(cf_row_unscaled, threshold_percentile=95):
    """CF point ki distance nearest 5 real patients se nikalo"""
    cf_scaled = scaler.transform(cf_row_unscaled.values.reshape(1, -1))
    distances, _ = nn_model.kneighbors(cf_scaled)
    avg_dist = distances.mean()
    return avg_dist

# Threshold set karo — training data ke apne andar ka average kNN distance
train_scaled = scaler.transform(X_train)
self_distances, _ = nn_model.kneighbors(train_scaled)
baseline_dist = self_distances[:, 1:].mean()  # khud ko exclude karo (0th neighbor)
dist_threshold = np.percentile(self_distances[:, 1:].mean(axis=1), 95)

print(f"Baseline average kNN distance (real data): {baseline_dist:.4f}")
print(f"95th percentile threshold: {dist_threshold:.4f}")

# ----------------------------------------------
# 2️⃣ GEOMETRIC CONSISTENCY CHECK — Radius/Perimeter/Area logical hain?
# ----------------------------------------------
def check_geometric_consistency(row):
    """
    Cell geometry check: area aur perimeter, radius ke proportional hone chahiye
    (roughly circular cell assumption — WBCD mein yeh measurements isi tarah nikalti hain)
    """
    checks = {}
    # Mean features
    expected_area_mean = np.pi * (row['radius_mean'])**2
    actual_area_mean = row['area_mean']
    checks['area_mean_ratio'] = actual_area_mean / expected_area_mean if expected_area_mean > 0 else np.nan

    expected_perimeter_mean = 2 * np.pi * row['radius_mean']
    actual_perimeter_mean = row['perimeter_mean']
    checks['perimeter_mean_ratio'] = actual_perimeter_mean / expected_perimeter_mean if expected_perimeter_mean > 0 else np.nan

    # Worst features
    expected_area_worst = np.pi * (row['radius_worst'])**2
    actual_area_worst = row['area_worst']
    checks['area_worst_ratio'] = actual_area_worst / expected_area_worst if expected_area_worst > 0 else np.nan

    return checks

# Training data pe "normal" ratio range nikalo (baseline)
train_ratios = X_train.apply(check_geometric_consistency, axis=1, result_type='expand')
ratio_bounds = {
    col: (train_ratios[col].quantile(0.01), train_ratios[col].quantile(0.99))
    for col in train_ratios.columns
}
print("\nGeometric ratio acceptable bounds (from real data):")
for k, v in ratio_bounds.items():
    print(f"  {k}: [{v[0]:.3f}, {v[1]:.3f}]")

# ----------------------------------------------
# 3️⃣ HAR PATIENT KE CFs PE PLAUSIBILITY CHECK LAGAO
# ----------------------------------------------
plausibility_results = []

for idx in high_uncertainty_indices:
    query_instance = X_test.iloc[[idx]]

    try:
        cf_df = cf_cache.get(idx)   # ← cache se (dobara generate nahi)

        if cf_df is None or len(cf_df) == 0:
            continue

        for cf_i, cf_row in cf_df.iterrows():
            cf_features = cf_row[feature_names]

            # Check 1: kNN distance
            knn_dist = check_knn_plausibility(cf_features)
            is_knn_plausible = knn_dist <= dist_threshold

            # Check 2: Geometric consistency
            geo_ratios = check_geometric_consistency(cf_features)
            is_geo_plausible = all(
                ratio_bounds[k][0] <= v <= ratio_bounds[k][1]
                for k, v in geo_ratios.items() if not np.isnan(v)
            )

            plausibility_results.append({
                'patient_idx': idx,
                'cf_num': cf_i,
                'knn_distance': knn_dist,
                'knn_plausible': is_knn_plausible,
                'geo_plausible': is_geo_plausible,
                'overall_plausible': is_knn_plausible and is_geo_plausible
            })
    except Exception as e:
        print(f"Patient {idx} error: {e}")

# ----------------------------------------------
# SUMMARY
# ----------------------------------------------
df_plausibility = pd.DataFrame(plausibility_results)
print(f"\n{'='*60}")
print("PLAUSIBILITY FILTERING SUMMARY")
print('='*60)
print(f"Total CFs evaluated: {len(df_plausibility)}")
print(f"kNN-plausible: {df_plausibility['knn_plausible'].sum()} ({df_plausibility['knn_plausible'].mean()*100:.1f}%)")
print(f"Geometrically plausible: {df_plausibility['geo_plausible'].sum()} ({df_plausibility['geo_plausible'].mean()*100:.1f}%)")
print(f"Overall plausible (both checks): {df_plausibility['overall_plausible'].sum()} ({df_plausibility['overall_plausible'].mean()*100:.1f}%)")

print("\nPer-patient plausibility:")
print(df_plausibility.groupby('patient_idx')['overall_plausible'].agg(['sum', 'count']))

In [ ]:
# ============================================
# STEP 4.7 (FINAL) — FILTER ONLY PLAUSIBLE CFs
# ============================================

import pandas as pd
import numpy as np

final_plausible_cfs = []

for idx in high_uncertainty_indices:
    query_instance = X_test.iloc[[idx]]
    true_label = y_test.iloc[idx]

    try:
        cf_df = cf_cache.get(idx)   # ← cache se (dobara generate nahi)

        if cf_df is None or len(cf_df) == 0:
            continue

        original_vals = query_instance.iloc[0]

        for cf_i, cf_row in cf_df.iterrows():
            cf_features = cf_row[feature_names]

            # Plausibility checks
            knn_dist = check_knn_plausibility(cf_features)
            is_knn_plausible = knn_dist <= dist_threshold

            geo_ratios = check_geometric_consistency(cf_features)
            is_geo_plausible = all(
                ratio_bounds[k][0] <= v <= ratio_bounds[k][1]
                for k, v in geo_ratios.items() if not np.isnan(v)
            )

            overall_plausible = is_knn_plausible and is_geo_plausible

            # ✅ SIRF PLAUSIBLE CFs SAVE KARO
            if overall_plausible:
                # Kaunsi features change hui, dhoondo
                changed_feats = {}
                for feat in feature_names:
                    orig_val = original_vals[feat]
                    cf_val = cf_row[feat]
                    if abs(cf_val - orig_val) > 0.001 * (permitted_range[feat][1] - permitted_range[feat][0]):
                        changed_feats[feat] = {'original': round(orig_val, 4), 'counterfactual': round(cf_val, 4)}

                final_plausible_cfs.append({
                    'patient_idx': idx,
                    'true_label': true_label,
                    'cf_number': cf_i,
                    'knn_distance': round(knn_dist, 4),
                    'n_features_changed': len(changed_feats),
                    'changed_features': changed_feats
                })
    except Exception as e:
        print(f"Patient {idx} error: {e}")

# ============================================
# FINAL CLEAN TABLE
# ============================================
df_final_plausible = pd.DataFrame(final_plausible_cfs)

print(f"{'='*60}")
print("FINAL FILTERED RESULTS — ONLY PLAUSIBLE COUNTERFACTUALS")
print('='*60)
print(f"Total plausible CFs retained: {len(df_final_plausible)}")
print(f"Patients with at least 1 plausible CF: {df_final_plausible['patient_idx'].nunique()}/12")

print("\n📋 Summary Table:")
print(df_final_plausible[['patient_idx', 'true_label', 'cf_number', 'knn_distance', 'n_features_changed']])

# Detailed view — har plausible CF ki exact changes dikhao
print(f"\n{'='*60}")
print("DETAILED VIEW — Feature Changes Per Plausible CF")
print('='*60)
for _, row in df_final_plausible.iterrows():
    print(f"\nPatient {row['patient_idx']} (True: {'Malignant' if row['true_label']==1 else 'Benign'}) — CF #{row['cf_number']}:")
    for feat, vals in row['changed_features'].items():
        print(f"  {feat}: {vals['original']} → {vals['counterfactual']}")

# Save for later use
print(f"\n✅ Saved as: df_final_plausible (variable ready for paper/visualization)")


## STEP 7 — L1 proximity + most-changed features (STEP 4.8)

In [ ]:
# ============================================
# L1 PROXIMITY SCORE COMPUTATION
# ============================================

import numpy as np
import pandas as pd

def compute_l1_proximity(original_row, cf_row, feature_names, permitted_range):
    """
    Normalized L1 distance — har feature ka change uske range se divide karo,
    phir sab ka sum lo.
    Lower score = closer/better counterfactual (kam changes)
    """
    total_distance = 0
    feature_distances = {}

    for feat in feature_names:
        orig_val = original_row[feat]
        cf_val = cf_row[feat]

        feat_range = permitted_range[feat][1] - permitted_range[feat][0]
        normalized_change = abs(cf_val - orig_val) / feat_range if feat_range > 0 else 0

        feature_distances[feat] = normalized_change
        total_distance += normalized_change

    # Average bhi nikalo (per-feature average distance) — interpretability ke liye
    avg_distance = total_distance / len(feature_names)

    return total_distance, avg_distance, feature_distances


# ============================================
# HAR PLAUSIBLE CF KE LIYE PROXIMITY SCORE NIKALO
# ============================================

proximity_results = []

for idx in high_uncertainty_indices:
    query_instance = X_test.iloc[[idx]]
    true_label = y_test.iloc[idx]
    original_vals = query_instance.iloc[0]

    try:
        cf_df = cf_cache.get(idx)   # ← cache se (dobara generate nahi)

        if cf_df is None or len(cf_df) == 0:
            continue

        for cf_i, cf_row in cf_df.iterrows():
            cf_features = cf_row[feature_names]

            # Plausibility check (pichla step)
            knn_dist = check_knn_plausibility(cf_features)
            is_knn_plausible = knn_dist <= dist_threshold
            geo_ratios = check_geometric_consistency(cf_features)
            is_geo_plausible = all(
                ratio_bounds[k][0] <= v <= ratio_bounds[k][1]
                for k, v in geo_ratios.items() if not np.isnan(v)
            )
            overall_plausible = is_knn_plausible and is_geo_plausible

            # ✅ L1 Proximity score nikalo
            total_dist, avg_dist, feat_dists = compute_l1_proximity(
                original_vals, cf_features, feature_names, permitted_range
            )

            proximity_results.append({
                'patient_idx': idx,
                'true_label': true_label,
                'cf_number': cf_i,
                'l1_proximity_total': round(total_dist, 4),
                'l1_proximity_avg': round(avg_dist, 4),
                'n_features_changed': sum(1 for v in feat_dists.values() if v > 0.001),
                'plausible': overall_plausible
            })
    except Exception as e:
        print(f"Patient {idx} error: {e}")

# ============================================
# RESULTS TABLE
# ============================================
df_proximity = pd.DataFrame(proximity_results)

print(f"{'='*60}")
print("L1 PROXIMITY SCORE — RESULTS")
print('='*60)
print(df_proximity[['patient_idx', 'cf_number', 'l1_proximity_total',
                      'l1_proximity_avg', 'n_features_changed', 'plausible']].to_string(index=False))

print(f"\n{'='*60}")
print("SUMMARY STATISTICS")
print('='*60)
print(f"Mean L1 proximity (all CFs):       {df_proximity['l1_proximity_total'].mean():.4f}")
print(f"Mean L1 proximity (plausible only): {df_proximity[df_proximity['plausible']]['l1_proximity_total'].mean():.4f}")
print(f"Min L1 proximity (best CF overall): {df_proximity['l1_proximity_total'].min():.4f}")
print(f"  → Patient: {df_proximity.loc[df_proximity['l1_proximity_total'].idxmin(), 'patient_idx']}")

print(f"\nBest (lowest distance) plausible CF per patient:")
best_per_patient = df_proximity[df_proximity['plausible']].sort_values('l1_proximity_total').groupby('patient_idx').first()
print(best_per_patient[['cf_number', 'l1_proximity_total', 'n_features_changed']])

In [ ]:
# ============================================
# STEP 4.8 — MOST CHANGED FEATURES ANALYSIS
# ============================================

import pandas as pd
import numpy as np
from collections import defaultdict

feature_frequency = defaultdict(int)       # kitni baar change hui
feature_magnitude = defaultdict(list)       # jab change hui, kitna bara tha
feature_plausible_frequency = defaultdict(int)  # sirf plausible CFs mein kitni baar

total_cfs_processed = 0

for idx in high_uncertainty_indices:
    query_instance = X_test.iloc[[idx]]
    original_vals = query_instance.iloc[0]

    try:
        cf_df = cf_cache.get(idx)   # ← cache se (dobara generate nahi)

        if cf_df is None or len(cf_df) == 0:
            continue

        for cf_i, cf_row in cf_df.iterrows():
            cf_features = cf_row[feature_names]
            total_cfs_processed += 1

            # Plausibility check
            knn_dist = check_knn_plausibility(cf_features)
            is_knn_plausible = knn_dist <= dist_threshold
            geo_ratios = check_geometric_consistency(cf_features)
            is_geo_plausible = all(
                ratio_bounds[k][0] <= v <= ratio_bounds[k][1]
                for k, v in geo_ratios.items() if not np.isnan(v)
            )
            overall_plausible = is_knn_plausible and is_geo_plausible

            # Har feature ka change check karo
            for feat in feature_names:
                orig_val = original_vals[feat]
                cf_val = cf_row[feat]
                feat_range = permitted_range[feat][1] - permitted_range[feat][0]
                normalized_change = abs(cf_val - orig_val) / feat_range if feat_range > 0 else 0

                if normalized_change > 0.001:  # significant change
                    feature_frequency[feat] += 1
                    feature_magnitude[feat].append(normalized_change)
                    if overall_plausible:
                        feature_plausible_frequency[feat] += 1
    except Exception as e:
        print(f"Patient {idx} error: {e}")

# ============================================
# SUMMARY TABLE BANAO
# ============================================
summary_data = []
for feat in feature_names:
    freq = feature_frequency.get(feat, 0)
    plaus_freq = feature_plausible_frequency.get(feat, 0)
    mags = feature_magnitude.get(feat, [])
    avg_mag = np.mean(mags) if mags else 0

    if freq > 0:
        summary_data.append({
            'feature': feat,
            'frequency': freq,
            'pct_of_cfs': round(freq / total_cfs_processed * 100, 1),
            'plausible_frequency': plaus_freq,
            'avg_normalized_change': round(avg_mag, 4)
        })

df_feature_analysis = pd.DataFrame(summary_data).sort_values('frequency', ascending=False)

print(f"{'='*70}")
print(f"MOST CHANGED FEATURES ANALYSIS (Total CFs analyzed: {total_cfs_processed})")
print('='*70)
print(df_feature_analysis.to_string(index=False))

print(f"\n{'='*70}")
print("TOP 5 — MOST FREQUENTLY CHANGED FEATURES")
print('='*70)
for _, row in df_feature_analysis.head(5).iterrows():
    print(f"  {row['feature']}: {row['frequency']}/{total_cfs_processed} CFs ({row['pct_of_cfs']}%) | avg change = {row['avg_normalized_change']}")

print(f"\n{'='*70}")
print("TOP 5 — LARGEST AVERAGE CHANGE (when changed)")
print('='*70)
df_by_magnitude = df_feature_analysis.sort_values('avg_normalized_change', ascending=False)
for _, row in df_by_magnitude.head(5).iterrows():
    print(f"  {row['feature']}: avg change = {row['avg_normalized_change']} | frequency = {row['frequency']}")

## STEP 8 — Figures (Fig 13, Fig 14)

In [ ]:
# ============================================
# STEP 4.9 — BEFORE/AFTER VISUALIZATION
# ============================================

import matplotlib.pyplot as plt
import numpy as np

# ----------------------------------------------
# Best counterfactual DYNAMICALLY nikalo
# ----------------------------------------------
best_row = (df_proximity[df_proximity['plausible']]
            .sort_values('l1_proximity_total')
            .iloc[0])
best_patient_idx = int(best_row['patient_idx'])
best_cf_number   = int(best_row['cf_number'])

query_instance = X_test.iloc[[best_patient_idx]]
original_vals  = query_instance.iloc[0]

print(f"Selected Patient {best_patient_idx}, CF #{best_cf_number} "
      f"(L1={best_row['l1_proximity_total']:.4f}, plausible=True)")

# cache se lo agar mojood ho, warna generate (fallback)
_best_idx = best_patient_idx if 'best_patient_idx' in dir() else high_uncertainty_indices[0]
if 'cf_cache' in dir() and cf_cache.get(_best_idx) is not None:
    cf_df_best = cf_cache[_best_idx]
else:
    cf_result_best = exp_dice.generate_counterfactuals(
    query_instance,
    total_CFs=3,
    desired_class="opposite",
    permitted_range=permitted_range,
    features_to_vary=features_to_vary
)
    cf_df_best = cf_result_best.cf_examples_list[0].final_cfs_df

# Sabse pehla CF lo (jo humne pehle best paya tha)
best_cf_row = cf_df_best.iloc[best_cf_number]

# Changed features dhoondo
changed_feats = {}
for feat in feature_names:
    orig_val = original_vals[feat]
    cf_val = best_cf_row[feat]
    feat_range = permitted_range[feat][1] - permitted_range[feat][0]
    if abs(cf_val - orig_val) / feat_range > 0.001:
        changed_feats[feat] = (orig_val, cf_val)

print(f"Patient {best_patient_idx} — Changed features: {changed_feats}")

# ----------------------------------------------
# PLOT 1 — Single Patient Before/After Bar Chart
# ----------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))

feats = list(changed_feats.keys())
originals = [changed_feats[f][0] for f in feats]
counterfactuals = [changed_feats[f][1] for f in feats]

x = np.arange(len(feats))
width = 0.35

bars1 = ax.bar(x - width/2, originals, width, label='Original (Malignant)', color='#E74C3C')
bars2 = ax.bar(x + width/2, counterfactuals, width, label='Counterfactual (Benign)', color='#3498DB')

ax.set_xlabel('Feature')
ax.set_ylabel('Value')
ax.set_title(f'Patient {best_patient_idx} — Before vs After Counterfactual\n(Minimal change needed to flip prediction)')
ax.set_xticks(x)
ax.set_xticklabels(feats, rotation=20, ha='right')
ax.legend()

# Value labels upar dikhao
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}', xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('patient_best_before_after.png', dpi=150, bbox_inches='tight')
plt.show()

# ----------------------------------------------
# PLOT 2 — Multi-Patient Summary: Top Changed Features (Frequency Bar Chart)
# ----------------------------------------------
fig2, ax2 = plt.subplots(figsize=(9, 5))

top_features = df_feature_analysis.head(8)
bars = ax2.barh(top_features['feature'], top_features['frequency'], color='#9B59B6')
ax2.set_xlabel('Number of Counterfactuals (out of 36)')
ax2.set_title('Most Frequently Changed Features Across 12 High-Uncertainty Patients')
ax2.invert_yaxis()  # top feature upar dikhe

for bar in bars:
    width = bar.get_width()
    ax2.annotate(f'{int(width)}', xy=(width, bar.get_y() + bar.get_height()/2),
                 xytext=(3, 0), textcoords="offset points", va='center', fontsize=9)

plt.tight_layout()
plt.savefig('feature_frequency_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Both visualizations saved: 'patient_best_before_after.png' and 'feature_frequency_chart.png'")

## STEP 9 — ★ RESULTS SUMMARY (sirf is cell ka output bhejein)

In [ ]:
import numpy as np, pandas as pd
rows = []
for idx in high_uncertainty_indices:
    cf_df = cf_cache[idx]; orig = X_test.iloc[idx]
    p_orig = float(ensemble_wrapper.predict_proba(X_test.iloc[[idx]])[0][1])
    for j, r in cf_df.iterrows():
        f = r[feature_names]; pr = df_proximity[(df_proximity.patient_idx == idx) & (df_proximity.cf_number == j)].iloc[0]
        changed = {k: (round(float(orig[k]),4), round(float(f[k]),4)) for k in feature_names
                   if abs(f[k]-orig[k]) / (permitted_range[k][1]-permitted_range[k][0]) > 0.001}
        rows.append(dict(patient_idx=idx, true_label=int(y_test.iloc[idx]), ens_prob_original=round(p_orig,4),
                         cf_number=j, cf_prob=round(float(ensemble_wrapper.predict_proba(f.to_frame().T)[0][1]),4),
                         l1=pr.l1_proximity_total, plausible=bool(pr.plausible), changes=changed))
all_cf = pd.DataFrame(rows); all_cf.to_csv('O4_all_counterfactuals.csv', index=False)

n = len(all_cf); npl = int(all_cf.plausible.sum())
print("="*72 + "\nDiCE RESULTS (seed-locked)\n" + "="*72)
print(f"Reproducible across two runs: {'✅' if DICE_REPRO else '❌'}")
print(f"Counterfactuals: {n} for {all_cf.patient_idx.nunique()} patients")
print(f"kNN-plausible {df_plausibility.knn_plausible.sum()} ({df_plausibility.knn_plausible.mean()*100:.1f}%) | "
      f"geometric {df_plausibility.geo_plausible.sum()} ({df_plausibility.geo_plausible.mean()*100:.1f}%) | "
      f"both {npl} ({npl/n*100:.1f}%)")
print(f"Patients with >=1 plausible CF: {all_cf[all_cf.plausible].patient_idx.nunique()}/12")
print(f"Mean L1: all {df_proximity.l1_proximity_total.mean():.4f} | plausible {df_proximity[df_proximity.plausible].l1_proximity_total.mean():.4f}")
print("\nTop-5 most frequently changed features (over all CFs):")
for _, r in df_feature_analysis.head(5).iterrows():
    print(f"   {r['feature']:<24} {r['frequency']}/{total_cfs_processed} ({r['pct_of_cfs']}%)")
b = all_cf[all_cf.plausible].sort_values('l1').iloc[0]
print(f"\nMost proximal plausible CF: Patient {b.patient_idx} (true {'M' if b.true_label else 'B'}), CF #{b.cf_number}, "
      f"L1 = {b.l1:.4f}, ensemble prob {b.ens_prob_original} -> {b.cf_prob}")
for k, (o, c) in b.changes.items():
    print(f"   {k}: {o} -> {c}  ({(c-o)/o*100:+.1f}%)" if o else f"   {k}: {o} -> {c}")
thr = best_ens_thr
all_cf['flips_at_thr'] = [(r.ens_prob_original >= thr) != (r.cf_prob >= thr) for r in all_cf.itertuples()]
all_cf.to_csv('O4_all_counterfactuals.csv', index=False)
print(f"\nCFs that flip the ensemble decision at its {thr:.2f} threshold: {int(all_cf.flips_at_thr.sum())}/{n}")
print(f"\nPer-patient (original ensemble prob, threshold {thr:.2f}):")
for idx in high_uncertainty_indices:
    s = all_cf[all_cf.patient_idx == idx]
    p = s.ens_prob_original.iloc[0]
    note = "  (0.35–0.5: pehle DiCE isay ghalat class samajhta)" if thr <= p < 0.5 else ""
    print(f"   P{idx:<4} true {'M' if s.true_label.iloc[0] else 'B'} | ens {p:.4f} | plausible {int(s.plausible.sum())}/{len(s)}{note}")


## STEP 10 — Figures + CSVs zip download

In [ ]:
import glob, zipfile
fs = sorted(set(glob.glob('*.png') + glob.glob('O4_*.csv')))
with zipfile.ZipFile('File4_results.zip', 'w') as z:
    for f in fs: z.write(f)
print(f"{len(fs)} files zipped:"); [print('  ', f) for f in fs]
print("\nPaper: Fig 13 -> feature_frequency_chart.png | Fig 14 -> patient_best_before_after.png")
try:
    from google.colab import files; files.download('File4_results.zip')
except Exception:
    print("Left side Files panel se File4_results.zip download karein.")
